# Simple MLP with JAX + Flax (NNX)

A 2-hidden-layer MLP classifying the sklearn two-moons dataset.

In [ ]:
import jax
import jax.numpy as jnp
import optax
from flax import nnx
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

print("JAX devices:", jax.devices())

## Data

In [ ]:
# Data: 2D two-moons binary classification
X, y = make_moons(n_samples=2000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test = jnp.asarray(X_train, jnp.float32), jnp.asarray(X_test, jnp.float32)
y_train, y_test = jnp.asarray(y_train), jnp.asarray(y_test)

## Model

NNX modules are plain Python objects: layers are attributes, `__call__` is the forward pass.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int, *, rngs: nnx.Rngs):
        self.fc1 = nnx.Linear(in_dim, hidden, rngs=rngs)
        self.fc2 = nnx.Linear(hidden, hidden, rngs=rngs)
        self.out = nnx.Linear(hidden, out_dim, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        x = nnx.relu(self.fc1(x))
        x = nnx.relu(self.fc2(x))
        return self.out(x)  # logits


model = MLP(in_dim=2, hidden=64, out_dim=2, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

## Loss, train step, and metrics

`nnx.jit` compiles the step; `nnx.value_and_grad` differentiates w.r.t. the model's params.

In [ ]:
def loss_fn(model: MLP, x: jax.Array, y: jax.Array) -> jax.Array:
    logits = model(x)
    return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()


@nnx.jit
def train_step(model: MLP, optimizer: nnx.Optimizer, x: jax.Array, y: jax.Array) -> jax.Array:
    loss, grads = nnx.value_and_grad(loss_fn)(model, x, y)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def accuracy(model: MLP, x: jax.Array, y: jax.Array) -> jax.Array:
    return (model(x).argmax(axis=-1) == y).mean()

## Train

In [ ]:
batch_size = 64
key = jax.random.key(1)
for epoch in range(1, 21):
    key, subkey = jax.random.split(key)
    perm = jax.random.permutation(subkey, len(X_train))
    for i in range(0, len(X_train), batch_size):
        idx = perm[i : i + batch_size]
        loss = train_step(model, optimizer, X_train[idx], y_train[idx])
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss {loss:.4f}  "
              f"train acc {accuracy(model, X_train, y_train):.3f}  "
              f"test acc {accuracy(model, X_test, y_test):.3f}")